# Hafta 11: Metin Madenciliği ve NLP

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Metin ön işleme (lowercase, noktalama temizleme, stopword)
2. Tokenization, stemming ve lemmatization örnekleri
3. Bag of Words ve TF-IDF temsili
4. Duygu analizi: sözlük tabanlı yaklaşım
5. Duygu analizi: makine öğrenmesi tabanlı yaklaşım
6. Konu modelleme (LDA)
7. Word2Vec ile kelime gömme
8. Sonuç ve karşılaştırma

## 1. Kütüphaneler

In [ ]:
import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.decomposition import LatentDirichletAllocation

# Opsiyonel paketler
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import SnowballStemmer
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

try:
    import spacy
    HAS_SPACY = True
except Exception:
    HAS_SPACY = False

try:
    from gensim.models import Word2Vec
    HAS_GENSIM = True
except Exception:
    HAS_GENSIM = False

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

print('NLTK  :', 'Hazır' if HAS_NLTK else 'Eksik')
print('spaCy :', 'Hazır' if HAS_SPACY else 'Eksik')
print('gensim:', 'Hazır' if HAS_GENSIM else 'Eksik')

## 2. Veri Yükleme

Ana veri seti: `twitter_sentiment.csv`
- `text`: tweet metni
- `sentiment`: `positive`, `negative`, `neutral`

In [ ]:
DATA_PATH = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', '..', 'Veri_Setleri_Genel', 'twitter_sentiment.csv'
)

df = pd.read_csv(DATA_PATH)
print(f'Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun')
display(df.head())

In [ ]:
print('Eksik değerler:')
print(df.isnull().sum())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='sentiment', palette='Set2')
plt.title('Sınıf Dağılımı')
plt.xlabel('Duygu')
plt.ylabel('Adet')
plt.tight_layout()
plt.show()

## 3. Metin Ön İşleme

Adımlar:
1. Küçük harfe çevirme
2. URL, mention ve hashtag temizleme
3. Noktalama ve sayı temizleme
4. Stopword kaldırma

In [ ]:
# İngilizce stopword listesi (NLTK yoksa basit fallback)
if HAS_NLTK:
    try:
        _ = stopwords.words('english')
    except LookupError:
        nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))
else:
    stop_words = {
        'a','an','the','and','or','is','are','to','of','in','on','for','with','this','that',
        'it','i','you','we','they','he','she','was','were','be','been','am','so','very'
    }

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)
display(df[['text', 'clean_text', 'sentiment']].head(10))

In [ ]:
# Tokenization, Stemming, Lemmatization örneği
sample_text = df.loc[0, 'text']
sample_clean = preprocess_text(sample_text)
sample_tokens = sample_clean.split()

print('Orijinal   :', sample_text)
print('Temiz      :', sample_clean)
print('Tokenlar   :', sample_tokens)

# Stemming (varsa)
if HAS_NLTK:
    stemmer = SnowballStemmer('english')
    stemmed = [stemmer.stem(w) for w in sample_tokens]
    print('Stemming   :', stemmed)
else:
    print('Stemming   : NLTK yüklü değil')

# Lemmatization (spaCy varsa)
if HAS_SPACY:
    try:
        nlp = spacy.load('en_core_web_sm')
        doc = nlp(sample_clean)
        lemmas = [tok.lemma_ for tok in doc]
        print('Lemmatize  :', lemmas)
    except Exception:
        print('Lemmatize  : spaCy modeli yok (python -m spacy download en_core_web_sm)')
else:
    print('Lemmatize  : spaCy yüklü değil')

## 4. Metin Temsili: BoW ve TF-IDF

In [ ]:
# BoW
bow_vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 2))
X_bow = bow_vectorizer.fit_transform(df['clean_text'])

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

print('BoW boyutu   :', X_bow.shape)
print('TF-IDF boyutu:', X_tfidf.shape)

print('\nBoW örnek özellikler:')
print(bow_vectorizer.get_feature_names_out()[:20])

print('\nTF-IDF örnek özellikler:')
print(tfidf_vectorizer.get_feature_names_out()[:20])

## 5. Duygu Analizi - Sözlük Tabanlı Yaklaşım

In [ ]:
positive_words = {'love', 'amazing', 'happy', 'best', 'incredible', 'great', 'blessed', 'excited'}
negative_words = {'worst', 'hate', 'annoying', 'terrible', 'bad', 'crashing', 'delayed'}

def lexicon_sentiment(text):
    words = text.split()
    pos = sum(1 for w in words if w in positive_words)
    neg = sum(1 for w in words if w in negative_words)
    score = pos - neg
    if score > 0:
        return 'positive'
    elif score < 0:
        return 'negative'
    else:
        return 'neutral'

df['lexicon_pred'] = df['clean_text'].apply(lexicon_sentiment)
lex_acc = accuracy_score(df['sentiment'], df['lexicon_pred'])
print(f'Sözlük tabanlı doğruluk: {lex_acc:.3f}')
display(df[['clean_text', 'sentiment', 'lexicon_pred']].head(12))

## 6. Duygu Analizi - Makine Öğrenmesi Tabanlı Yaklaşım

Modeller:
- Multinomial Naive Bayes (TF-IDF ile)
- Logistic Regression (TF-IDF ile)

In [ ]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Eğitim boyutu:', len(X_train))
print('Test boyutu  :', len(X_test))

In [ ]:
# Naive Bayes pipeline
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1500, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Naive Bayes doğruluk: {acc_nb:.3f}')
print('\nSınıflandırma raporu (NB):')
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1500, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression doğruluk: {acc_lr:.3f}')
print('\nSınıflandırma raporu (LR):')
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Karşılaştırma tablosu
comparison = pd.DataFrame({
    'Yöntem': ['Lexicon', 'Naive Bayes + TF-IDF', 'Logistic Regression + TF-IDF'],
    'Accuracy': [lex_acc, acc_nb, acc_lr]
})

display(comparison.sort_values('Accuracy', ascending=False).reset_index(drop=True))

plt.figure(figsize=(8, 4))
sns.barplot(data=comparison, x='Yöntem', y='Accuracy', palette='viridis')
plt.ylim(0, 1.0)
plt.title('Duygu Analizi Yaklaşımları - Doğruluk Karşılaştırması')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# En iyi model için confusion matrix
best_name = comparison.sort_values('Accuracy', ascending=False).iloc[0]['Yöntem']

if best_name == 'Naive Bayes + TF-IDF':
    y_best = y_pred_nb
elif best_name == 'Logistic Regression + TF-IDF':
    y_best = y_pred_lr
else:
    # Lexicon tüm veri üstünde hesaplandığı için test kesitine projekte edelim
    test_df = pd.DataFrame({'text': X_test, 'true': y_test})
    y_best = test_df['text'].apply(lambda x: lexicon_sentiment(preprocess_text(x)))

cm = confusion_matrix(y_test, y_best, labels=['negative', 'neutral', 'positive'])
disp = ConfusionMatrixDisplay(cm, display_labels=['negative', 'neutral', 'positive'])
disp.plot(cmap='Blues', colorbar=False)
plt.title(f'En iyi yöntem confusion matrix: {best_name}')
plt.tight_layout()
plt.show()

## 7. Konu Modelleme (LDA)

`twitter_sentiment.csv` küçük bir veri seti olduğu için örnek amaçlı uygulanmaktadır.

In [ ]:
cv = CountVectorizer(max_features=500, stop_words='english')
X_counts = cv.fit_transform(df['clean_text'])

n_topics = 3
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X_counts)

feature_names = cv.get_feature_names_out()
n_top_words = 8

for topic_idx, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-n_top_words:][::-1]
    top_words = [feature_names[i] for i in top_idx]
    print(f"Konu {topic_idx + 1}: {', '.join(top_words)}")

## 8. Word Embeddings (Word2Vec)

Gensim yüklüyse kısa bir Word2Vec örneği eğitilir.

In [ ]:
if HAS_GENSIM:
    tokenized_sentences = [t.split() for t in df['clean_text'] if isinstance(t, str) and len(t) > 0]

    w2v_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=50,
        window=3,
        min_count=1,
        workers=1,
        seed=42
    )

    sample_word = 'amazing'
    if sample_word in w2v_model.wv:
        print(f"'{sample_word}' için en benzer kelimeler:")
        print(w2v_model.wv.most_similar(sample_word, topn=5))
    else:
        print(f"'{sample_word}' kelimesi sözlükte yok.")

    print('\nVektör boyutu:', w2v_model.wv.vector_size)
else:
    print('Word2Vec için gensim gerekli: pip install gensim')

## 9. Ek Uygulama: IMDB Yorumlarında İkili Duygu Analizi

Aynı pipeline yaklaşımı farklı veri setine taşınır.

In [ ]:
IMDB_PATH = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', '..', 'Veri_Setleri_Genel', 'imdb_reviews.csv'
)

imdb = pd.read_csv(IMDB_PATH)
imdb['clean_review'] = imdb['review'].apply(preprocess_text)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    imdb['clean_review'], imdb['sentiment'], test_size=0.25, random_state=42, stratify=imdb['sentiment']
)

imdb_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

imdb_model.fit(X_train_i, y_train_i)
pred_i = imdb_model.predict(X_test_i)
acc_i = accuracy_score(y_test_i, pred_i)

print(f'IMDB doğruluk: {acc_i:.3f}')
print(classification_report(y_test_i, pred_i))

## 10. Sonuç

Bu uygulamada metin madenciliği sürecinin uçtan uca bir örneği gösterildi:
- Metin temizleme ve token düzeyinde hazırlık
- BoW ve TF-IDF ile sayısallaştırma
- Sözlük tabanlı ve makine öğrenmesi tabanlı duygu analizi
- LDA ile konu modelleme
- Word2Vec ile kelime gömme

Pratik öneri:
1. Önce basit bir TF-IDF + Naive Bayes baseline kurun.
2. Sonra Logistic Regression ve n-gram ile performansı artırın.
3. Daha büyük veri setlerinde transformer modellerine geçin.